# ISL low-data study — one notebook, four team members, no waiting on each other

Settings: **Accelerator = GPU T4 x2**, **Internet = on**.
Run with *Save Version → Save & Run All (Commit)*, which runs in the background for up to 12 h.

Every member runs the **same notebook** with their own `MEMBER` number (0–3):

| STAGE | what it does | inputs to attach (all optional) |
|---|---|---|
| `member` | **1. data**: builds the landmark stores the grids need, or copies them if a complete copy is attached. **2. sweep**: this member's share of every grid, on both T4s. | this notebook's previous version (to resume), and/or a teammate's output or a shared stores dataset (to skip extraction) |
| `report` | tables, plots and paired tests over all members | the members' final outputs |

Nobody waits for anyone. Each member builds the full data on their own; attached inputs only
save time. With the default grids (the 30-word corpus from Hugging Face, pinned to one commit),
the data step takes a few minutes. The report checks that all members used identical
data (same clip fingerprint).

Nothing here uploads anything. Stores and results stay in this notebook's output until
you choose to turn them into a (private) dataset.

In [ ]:
STAGE = "member"         # member | report
MEMBER = 0               # 0..3: each team member uses their own number
MEMBERS = 4
GRIDS = [                # run in this order; the first one is the main experiment
    "islr/lowdata/configs/isl40_scarce.json",
    "islr/lowdata/configs/isl40_uniform.json",
    "islr/lowdata/configs/isl40_vocab.json",
]
TIME_BUDGET_H = 11.3     # whole notebook; Kaggle stops at 12 h and runs checkpoint before this

# where the code comes from: a private Kaggle dataset with the repo, or a git branch
CODE_DATASET = "/kaggle/input/isl-lowdata-slr-code"   # used if it exists
REPO_URL = "https://github.com/Vidit-01/isl-lowdata-slr.git"
BRANCH = "main"

In [ ]:
import glob, json, os, shutil, subprocess, sys, time
T0 = time.time()
WORK = "/kaggle/working"
STORES = f"{WORK}/stores"          # built or copied by the data step; part of this notebook's output
SWEEPS = f"{WORK}/sweeps"
CODE = "/kaggle/tmp/code" if os.path.isdir("/kaggle/tmp") else "/tmp/code"
if os.path.isdir(CODE_DATASET):
    shutil.copytree(CODE_DATASET, CODE, dirs_exist_ok=True)
elif not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, CODE], check=True)
os.chdir(CODE)
os.environ["STORES"] = STORES
os.environ["PYTHONUNBUFFERED"] = "1"
try:  # Kaggle Secrets (Add-ons → Secrets): HF_TOKEN, only for gated/private Hugging Face data
    from kaggle_secrets import UserSecretsClient
    os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
except Exception:
    pass

def sh(*args, env=None):
    print("$", " ".join(map(str, args)), flush=True)
    r = subprocess.run([str(a) for a in args], env=env)
    print("exit code", r.returncode)
    return r.returncode

def hours_left():
    return TIME_BUDGET_H - (time.time() - T0) / 3600

def inputs_named(name):
    # folders called `name` in attached inputs (/kaggle/input/<x>/name, or deeper in the newer layout)
    return sorted({p for d in range(1, 4) for p in glob.glob("/kaggle/input/" + "*/" * d + name)
                   if os.path.isdir(p)})

import torch
print(torch.__version__, torch.cuda.device_count(), "GPU(s)",
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 1. Data (this member's own copy; nothing to wait for)

For every store the grids list, `lowdata.py data`:
1. keeps it if `/kaggle/working/stores/<name>` is already complete;
2. otherwise copies a complete copy with the same data version from `/kaggle/input`
   (your previous version, a teammate's output, or a shared stores dataset);
3. otherwise builds it. The 30-word corpus is downloaded from Hugging Face at the pinned commit
   and MediaPipe runs on the CPUs (a few minutes). Full INCLUDE, used only by the `scarce_legacy8*` grids,
   is ~57 GB from a slow Zenodo server: partial copies from teammates are merged and their
   finished zips skipped, and each member starts on a different part.

MediaPipe is installed into its own folder, and only the data subprocess sees it.
The store fingerprints go to `sweeps/_sweep/data_member<M>.json` for the report.

In [ ]:
if STAGE == "member":
    env = dict(os.environ, PYTHONPATH="/tmp/mp", HOLISTIC_TASK_PATH="/tmp/holistic_landmarker.task")
    if not os.path.isdir("/tmp/mp"):
        sh(sys.executable, "-m", "pip", "install", "-q", "--target", "/tmp/mp",
           "mediapipe==0.10.21", "opencv-python-headless")
    DATA_RC = sh(sys.executable, "lowdata.py", "data", "--grid", *GRIDS, "--dest", STORES,
                 "--member", MEMBER, "--members", MEMBERS, "--inputs", "/kaggle/input",
                 "--time-budget-h", round(hours_left() - 0.5, 2), "--workers", os.cpu_count(),
                 "--work", "/tmp/dl", env=env)
    os.makedirs(f"{SWEEPS}/_sweep", exist_ok=True)
    shutil.copy(f"{STORES}/_status.json", f"{SWEEPS}/_sweep/data_member{MEMBER}.json")
    print(open(f"{STORES}/_status.json").read())
    if DATA_RC != 0:
        print("DATA NOT COMPLETE: save this version, attach its output, and run again "
              "(or attach a teammate's output that has the stores)")

## 2. Sweep (this member's share)

1. quick GPU smoke test (2–3 min): every code path on the T4, with fp16 on;
2. `inspect`: store statistics, and whether every K works for every seed;
3. for each grid in turn, this member's share, with one job queue per T4 (see `sweep.py` for why
   these small jobs don't use DDP). Results go to `/kaggle/working/sweeps`. A job that appears in two grids
   runs only once.

**Resuming after 12 h:** add this notebook's previous version as an input
(Add Input → Your Work → this notebook), then run again. The stores are copied back, finished runs are
skipped, and interrupted runs continue from their checkpoint.

In [ ]:
if STAGE == "member" and DATA_RC == 0:
    sh(sys.executable, "lowdata.py", "smoke", "--fast", "--device", "cuda", "--skip", "ddp", "sweep")
    stores = sorted({os.path.expandvars(s) for g in GRIDS for s in json.load(open(g))["stores"]})
    sh(sys.executable, "lowdata.py", "inspect", "--stores", *stores, "--grid", GRIDS[0], "--dest", f"{WORK}/inspect")
    for g in GRIDS:
        sh(sys.executable, "lowdata.py", "sweep", "--grid", g, "--plan", "--members", MEMBERS)

In [ ]:
if STAGE == "member" and DATA_RC == 0:
    resume = inputs_named("sweeps")
    print("resuming from", resume)
    left = []
    for g in GRIDS:
        if hours_left() < 0.25:
            left.append(g)
            continue
        rc = sh(sys.executable, "lowdata.py", "sweep", "--grid", g, "--member", MEMBER, "--members", MEMBERS,
                "--out", SWEEPS, "--cache-dir", "/tmp/bank", "--time-budget-h", round(hours_left(), 2),
                *(["--resume-from", *resume] if resume else []))
        if rc != 0:
            left.append(g)
    print("ALL DONE" if not left else f"NOT FINISHED ({left}): save this version, attach its output, and run again")
    sh(sys.executable, "lowdata.py", "report", "--out", SWEEPS, "--dest", f"{SWEEPS}/_report")

## Stage `report`

Attach every member's final output as an input (their notebooks must be shared with you,
or published as private datasets you can see). The checks include whether every member
used identical data.

In [ ]:
if STAGE == "report":
    outs = inputs_named("sweeps")
    print(outs)
    sh(sys.executable, "lowdata.py", "report", "--out", *outs, "--dest", f"{WORK}/report")
    from IPython.display import Markdown, display
    display(Markdown(open(f"{WORK}/report/summary.md").read()))
    display(Markdown(open(f"{WORK}/report/checks.md").read()))